# 트레이닝/학습 데이터 분류 8:2 (overfitting 방지)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# [STEP 1] 데이터셋 생성 로직 (기존 부록 코드 기반 요약)
substances_info = {
    'Novec 4200': {'gamma': 17.0, 'theta': 115.0},
    'Triton BG-10': {'gamma': 28.0, 'theta': 75.0},
    'Triton CG-50': {'gamma': 27.0, 'theta': 78.0},
    'Brij S100': {'gamma': 56.0, 'theta': 65.0},
    'DI-03': {'gamma': 72.0, 'theta': 70.0}
} # [cite: 212, 213, 214, 215, 216, 217]

NUM_SAMPLES = 20000 # [cite: 218]
np.random.seed(42) # [cite: 219]

temp = np.random.uniform(20, 90, NUM_SAMPLES) # [cite: 221]
rpm = np.random.uniform(500, 10000, NUM_SAMPLES) # [cite: 222]
sub_names = np.random.choice(list(substances_info.keys()), NUM_SAMPLES) # [cite: 223]

gamma_base = np.array([substances_info[name]['gamma'] for name in sub_names]) # [cite: 225]
theta = np.array([substances_info[name]['theta'] for name in sub_names]) # [cite: 226, 228]

# 물리 엔진 연산 및 붕괴 판정 [cite: 229]
gamma_eff = gamma_base - 0.1 * (temp - 25.0) # [cite: 232]
P_cap = np.abs((2 * (gamma_eff * 1e-3) * np.cos(np.radians(theta))) / 20e-9) * 1e-6 # [cite: 235]
P_rpm = 0.00000008 * (rpm**2) # [cite: 236]
P_total = P_cap + P_rpm # [cite: 237]

damage_factor = 1.0 - 0.15 * np.random.randn(NUM_SAMPLES) # [cite: 239, 240]
P_limit = 5.0 * damage_factor # [cite: 241]
collapse = (P_total > P_limit).astype(int) # [cite: 242]

df = pd.DataFrame({
    'Temp': temp, 'RPM': rpm, 'Gamma': gamma_base, 'Theta': theta, 'Collapse': collapse
}) # [cite: 244, 245, 246, 247, 248, 249]

# ========================================================
# 💡 김민준 역할 핵심: 데이터 분리 및 다중 지표 검증 체계 도입
# ========================================================

# 1. 독립변수(X)와 종속변수(y) 설정 [cite: 254, 255]
X = df[['Temp', 'RPM', 'Gamma', 'Theta']] # [cite: 254]
y = df['Collapse'] # [cite: 255]

# 2. Train 데이터와 Test 데이터를 8:2 비율로 분리
# stratify=y 옵션은 학습과 테스트셋의 붕괴/안전 데이터 비율을 균일하게 맞춰줌
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"학습 데이터 개수: {X_train.shape[0]}개 | 테스트 데이터 개수: {X_test.shape[0]}개")

# 3. Random Forest 모델을 '학습 데이터(Train)'로만 학습 [cite: 256]
rf_model = RandomForestClassifier(n_estimators=100, random_state=42) # [cite: 256]
rf_model.fit(X_train, y_train)

# 4. '테스트 데이터(Test)'를 투입해 예측 성능 검증
y_pred = rf_model.predict(X_test)

# 5. 교수님 방어용 다각화 성능 지표 산출
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n[✔] Random Forest 검증 결과")
print(f"- Accuracy (전체 예측 정확도)  : {accuracy:.4f}")
print(f"- Precision (붕괴라 예측한 것 중 실제 붕괴) : {precision:.4f}")
print(f"- Recall (실제 붕괴 중 모델이 맞춘 비율)   : {recall:.4f}")
print(f"- F1-Score (정밀도와 재현율의 조화 평균)  : {f1:.4f}")

학습 데이터 개수: 16000개 | 테스트 데이터 개수: 4000개

[✔] Random Forest 검증 결과
- Accuracy (전체 예측 정확도)  : 0.9347
- Precision (붕괴라 예측한 것 중 실제 붕괴) : 0.9117
- Recall (실제 붕괴 중 모델이 맞춘 비율)   : 0.8950
- F1-Score (정밀도와 재현율의 조화 평균)  : 0.9033


# 데이터 생성함수(RandomForest,DecisionTree,GradientBooasting)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# ========================================================
# 💡 [피드백 2, 3번 반영] 유효 점도(viscosity) 및 신규 변수 추가 정의
# ========================================================
substances_info = {
    'Novec 4200': {'gamma': 17.0, 'theta': 115.0, 'viscosity': 1.02},     # [cite: 61, 212]
    'Triton BG-10': {'gamma': 28.0, 'theta': 75.0, 'viscosity': 1.15},   # [cite: 61, 212]
    'Triton CG-50': {'gamma': 27.0, 'theta': 78.0, 'viscosity': 1.20},   # [cite: 61, 213]
    'Brij S100': {'gamma': 56.0, 'theta': 65.0, 'viscosity': 1.50},      # [cite: 61, 216]
    'DI-03': {'gamma': 72.0, 'theta': 70.0, 'viscosity': 1.0}            # [cite: 61, 217]
}

NUM_SAMPLES = 20000
np.random.seed(42)

temp = np.random.uniform(20, 90, NUM_SAMPLES) # [cite: 221]
rpm = np.random.uniform(500, 10000, NUM_SAMPLES) # [cite: 222]
sub_names = np.random.choice(list(substances_info.keys()), NUM_SAMPLES) # [cite: 223]

# 💡 신규 변수: 웨이퍼 반지름 위치(r)를 0~150mm 사이의 확률적 변수로 생성
radius = np.random.uniform(0, 150, NUM_SAMPLES)

gamma_base = np.array([substances_info[name]['gamma'] for name in sub_names]) # [cite: 225]
theta = np.array([substances_info[name]['theta'] for name in sub_names]) # [cite: 226]
# 💡 신규 변수: 물질별 유효 점도 데이터 매핑
viscosity_base = np.array([substances_info[name]['viscosity'] for name in sub_names])

# [물리 엔진 업그레이드]
gamma_eff = gamma_base - 0.1 * (temp - 25.0) # [cite: 232]
P_cap = np.abs((2 * (gamma_eff * 1e-3) * np.cos(np.radians(theta))) / 20e-9) * 1e-6 # [cite: 235]

# 💡 피드백 반영 수식 고도화: P_rpm에 실제 반지름 변수(r / r_max) 동역학 반영
P_rpm = (radius / 150.0) * 0.00000008 * (rpm**2) # [cite: 69, 236]

# 💡 피드백 반영 수식 고도화: 유효 점도에 따른 유체 전단 응력(Shear Stress) 패널티 추가
P_viscous = viscosity_base * (rpm / 10000.0) * (radius / 150.0) * 0.15

# 최종 통합 압력 연산
P_total = P_cap + P_rpm + P_viscous

damage_factor = 1.0 - 0.15 * np.random.randn(NUM_SAMPLES) # [cite: 239]
P_limit = 5.0 * damage_factor # [cite: 241]
collapse = (P_total > P_limit).astype(int) # [cite: 242]

# 💡 변수가 6개로 늘어난 고차원 공정 데이터프레임 구축
df = pd.DataFrame({
    'Temp': temp, 'RPM': rpm, 'Gamma': gamma_base, 'Theta': theta,
    'Viscosity': viscosity_base, 'Radius': radius, 'Collapse': collapse
})

# 독립변수(X)에 신규 변수 포함 총 6개 지정
X = df[['Temp', 'RPM', 'Gamma', 'Theta', 'Viscosity', 'Radius']]
y = df['Collapse']

# Train / Test 세트 8:2 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ========================================================
# 💡 [피드백 4번 반영] 확장된 데이터셋 위에서 다중 모델 재검증
# ========================================================
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

benchmark_results = []

print("[진행 중] 고도화된 데이터셋 기반 모델별 학습 및 검증 수행...")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    benchmark_results.append({
        'Model': name, 'Accuracy': round(acc, 4), 'Precision': round(prec, 4),
        'Recall': round(rec, 4), 'F1-Score': round(f1, 4)
    })

perf_df = pd.DataFrame(benchmark_results)
print("\n=======================================================")
print("[✔] 최종 고도화 결과: 6개 변수 기준 모델 성능 비교 벤치마크")
print("=======================================================")
print(perf_df.to_string(index=False))
print("=======================================================")

[진행 중] 고도화된 데이터셋 기반 모델별 학습 및 검증 수행...

[✔] 최종 고도화 결과: 6개 변수 기준 모델 성능 비교 벤치마크
            Model  Accuracy  Precision  Recall  F1-Score
    Decision Tree    0.9407     0.7857  0.7543    0.7697
    Random Forest    0.9545     0.8565  0.7848    0.8191
Gradient Boosting    0.9597     0.8792  0.8038    0.8398


# Grid Search로 변수 최적값 도출

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import GradientBoostingClassifier

# [STEP 1] 실제 데이터 기반 물질 특성 정의
substances_info = {
    'Novec 4200': {'gamma': 17.0, 'theta': 115.0, 'viscosity': 1.02},
    'Triton BG-10': {'gamma': 28.0, 'theta': 75.0, 'viscosity': 1.15},
    'Triton CG-50': {'gamma': 27.0, 'theta': 78.0, 'viscosity': 1.20},
    'Brij S100':    {'gamma': 56.0, 'theta': 65.0, 'viscosity': 1.50},
    'DI-03':        {'gamma': 72.0, 'theta': 70.0, 'viscosity': 1.0}
}

NUM_SAMPLES = 20000
np.random.seed(42)

temp = np.random.uniform(20, 90, NUM_SAMPLES)
rpm = np.random.uniform(500, 6000, NUM_SAMPLES)
radius = np.random.uniform(0, 150, NUM_SAMPLES)

sub_names = np.random.choice(list(substances_info.keys()), NUM_SAMPLES)
gamma_base = np.array([substances_info[name]['gamma'] for name in sub_names])
theta = np.array([substances_info[name]['theta'] for name in sub_names])
viscosity_base = np.array([substances_info[name]['viscosity'] for name in sub_names])

# [물리 엔진 업그레이드]
gamma_eff = gamma_base - 0.1 * (temp - 25.0)
P_cap = np.abs((2 * (gamma_eff * 1e-3) * np.cos(np.radians(theta))) / 20e-9) * 1e-6
P_rpm = (radius / 150.0) * 0.00000008 * (rpm**2)
P_viscous = viscosity_base * (rpm / 6000.0) * (radius / 150.0) * 0.15

# ========================================================
# 💡 김민준의 칩샷: 실제 장비 기계적 진동 및 슬립 리스크 페널티 추가
# ========================================================
# 4,500 RPM 부근부터 기하급수적으로 증가하는 모터 구조 진동 응력 구현
P_vibration = 0.00000013 * (rpm ** 2) * (radius / 150.0)

# 최종 통합 압력 (패턴 붕괴 압력 + 유동 불균일 점도 응력 + 설비 진동 스트레스)
P_total = P_cap + P_rpm + P_viscous + P_vibration

damage_factor = 1.0 - 0.15 * np.random.randn(NUM_SAMPLES)
P_limit = 5.0 * damage_factor
collapse = (P_total > P_limit).astype(int)

df = pd.DataFrame({
    'Temp': temp, 'RPM': rpm, 'Gamma': gamma_base, 'Theta': theta,
    'Viscosity': viscosity_base, 'Radius': radius, 'Collapse': collapse
})

X = df[['Temp', 'RPM', 'Gamma', 'Theta', 'Viscosity', 'Radius']]
y = df['Collapse']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

best_model = GradientBoostingClassifier(random_state=42)
best_model.fit(X_train, y_train)

# ========================================================
# 💡 [3~4단계] 장비 제약 반영된 AI 모델 기반 Grid Search 전수 탐색
# ========================================================
process_constraints = {
    'Novec 4200':   {'min_temp': 20, 'max_temp': 90},
    'Triton BG-10': {'min_temp': 20, 'max_temp': 90},
    'Triton CG-50': {'min_temp': 20, 'max_temp': 90},
    'Brij S100':    {'min_temp': 55, 'max_temp': 90},
    'DI-03':        {'min_temp': 20, 'max_temp': 40}
}

final_recipes = []

for name, info in substances_info.items():
    limits = process_constraints[name]

    temp_range = np.arange(limits['min_temp'], limits['max_temp'] + 1, 1)
    rpm_range = np.arange(500, 6001, 100)

    T, R = np.meshgrid(temp_range, rpm_range)

    grid_df = pd.DataFrame({
        'Temp': T.flatten(),
        'RPM': R.flatten(),
        'Gamma': info['gamma'],
        'Theta': info['theta'],
        'Viscosity': info['viscosity'],
        'Radius': 150.0
    })

    grid_df['Collapse_Pred'] = best_model.predict(grid_df[['Temp', 'RPM', 'Gamma', 'Theta', 'Viscosity', 'Radius']])
    safe_zone = grid_df[grid_df['Collapse_Pred'] == 0]

    if not safe_zone.empty:
        best_recipe = safe_zone.sort_values(by='RPM', ascending=False).iloc[0]
        final_recipes.append({
            'Substance': name,
            'Opt_Temp (°C)': int(best_recipe['Temp']),
            'Max_Safe_RPM': int(best_recipe['RPM']),
            'Status': 'Success'
        })
    else:
        final_recipes.append({
            'Substance': name, 'Opt_Temp (°C)': '-', 'Max_Safe_RPM': '-', 'Status': 'Failed'
        })

recipe_df = pd.DataFrame(final_recipes)
print("\n=======================================================")
print("[✔] 장비 기계적 제약(진동/슬립) 최종 반영 최적 공정 레시피")
print("=======================================================")
print(recipe_df.to_string(index=False))
print("=======================================================")


[✔] 장비 기계적 제약(진동/슬립) 최종 반영 최적 공정 레시피
   Substance  Opt_Temp (°C)  Max_Safe_RPM  Status
  Novec 4200             60          4400 Success
Triton BG-10             62          4400 Success
Triton CG-50             63          4600 Success
   Brij S100             70          3500 Success
       DI-03             40          3200 Success


# 베이지안 최적화로 도출한 DI-O3 변수 최적값

In [ ]:
import pandas as pd
import numpy as np
from skopt import gp_minimize
from skopt.space import Integer
from skopt.utils import use_named_args
import warnings
warnings.filterwarnings('ignore')

# 💡 [원인 차단] 주피터 세션 꼬임 방지를 위해 딕셔너리 구조를 소문자 key로 명확히 고정
substances_info = {
    'Novec 4200': {'gamma': 17.0, 'theta': 115.0, 'viscosity': 1.02},
    'Triton BG-10': {'gamma': 28.0, 'theta': 75.0, 'viscosity': 1.15},
    'Triton CG-50': {'gamma': 27.0, 'theta': 78.0, 'viscosity': 1.20},
    'Brij S100':    {'gamma': 56.0, 'theta': 65.0, 'viscosity': 1.50},
    'DI-03':        {'gamma': 72.0, 'theta': 70.0, 'viscosity': 1.0}
}

target_substance = 'DI-03'
info = substances_info[target_substance]

# 1. 탐색 공간 정의 (Grid Search와 완벽히 동일한 DI-O3 제약 조건 설정)
search_space = [
    Integer(20, 40, name='Temp'),
    Integer(500, 6000, name='RPM')
]

# 2. 베이지안 최적화용 목적 함수 정의
@use_named_args(search_space)
def bayesian_objective(Temp, RPM):
    # AI 모델 투입용 6차원 입력 벡터 구축 (Worst-case 반지름 150mm 고정)
    input_data = pd.DataFrame([{
        'Temp': float(Temp),
        'RPM': float(RPM),
        'Gamma': info['gamma'],
        'Theta': info['theta'],
        'Viscosity': info['viscosity'], # 위에서 정의한 소문자 key와 완벽 매칭
        'Radius': 150.0
    }])

    # 앞서 학습 완료한 고도화된 AI 모델(Gradient Boosting)로 붕괴 예측 (0: Safe, 1: Collapse)
    pred = best_model.predict(input_data[['Temp', 'RPM', 'Gamma', 'Theta', 'Viscosity', 'Radius']])[0]

    if pred == 1:
        # 패턴이 붕괴하면 최악의 점수(0) 반환
        return 0.0
    else:
        # 안전하면 RPM을 극대화하기 위해 최소화 문제용 음수 변환
        return -float(RPM)

print(f"[진행 중] {target_substance} 기준 6개 변수 환경 베이지안 최적화 탐색 시작 (50회 샘플링)...")

# 3. 가우시안 프로세스 기반 베이지안 최적화 실행
res = gp_minimize(
    bayesian_objective,
    dimensions=search_space,
    n_calls=50,
    random_state=42,
    n_initial_points=10
)

# 4. 결과 도출 및 출력
opt_temp = res.x[0]
max_bayesian_rpm = -res.fun

print("\n=======================================================")
print("[✔] 동일 조건(6개 변수) 하의 베이지안 최적화 최종 결과")
print("=======================================================")
print(f"- 대상 약액 물질: {target_substance}")
print(f"- AI 도출 최적 온도: {opt_temp} °C")
print(f"- AI 도출 최대 안전 RPM: {int(max_bayesian_rpm)} RPM")
print("=======================================================")

[진행 중] DI-03 기준 6개 변수 환경 베이지안 최적화 탐색 시작 (50회 샘플링)...

[✔] 동일 조건(6개 변수) 하의 베이지안 최적화 최종 결과
- 대상 약액 물질: DI-03
- AI 도출 최적 온도: 20 °C
- AI 도출 최대 안전 RPM: 3059 RPM
